<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:
# DM4ML - Assignment - Validation + Auto-Fix + Revalidation + PDF Report

import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak

# ============================================================
# RecoMart Validation - robust file discovery + validation
# with remediation, revalidation, and PDF reporting
# ============================================================

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_ROOT = PROJECT_ROOT / "data" / "raw"
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze"

REMEDIATED_RAW_ROOT = PROJECT_ROOT / "data" / "remediated" / "raw"
REMEDIATED_BRONZE_ROOT = PROJECT_ROOT / "data" / "remediated" / "bronze"

REPORT_ROOT = PROJECT_ROOT / "reports" / "validation"
LOG_ROOT = PROJECT_ROOT / "logs"

for folder in [REPORT_ROOT, LOG_ROOT, REMEDIATED_RAW_ROOT, REMEDIATED_BRONZE_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

UTC_NOW = datetime.now(timezone.utc)
RUN_ID = UTC_NOW.strftime("%Y%m%dT%H%M%SZ")
RUN_TS = UTC_NOW.isoformat()

LOG_FILE = LOG_ROOT / f"validation_log_{RUN_ID}.jsonl"
INITIAL_ISSUES_FILE = REPORT_ROOT / f"validation_issues_initial_{RUN_ID}.csv"
INITIAL_SUMMARY_FILE = REPORT_ROOT / f"validation_summary_initial_{RUN_ID}.csv"
FINAL_ISSUES_FILE = REPORT_ROOT / f"validation_issues_revalidated_{RUN_ID}.csv"
FINAL_SUMMARY_FILE = REPORT_ROOT / f"validation_summary_revalidated_{RUN_ID}.csv"
FIX_LOG_FILE = REPORT_ROOT / f"fix_log_{RUN_ID}.csv"
REPORT_FILE = REPORT_ROOT / f"data_quality_report_{RUN_ID}.json"
PDF_REPORT_FILE = REPORT_ROOT / f"data_quality_report_{RUN_ID}.pdf"

ALLOWED_EVENT_TYPES = {"view", "addtocart", "transaction"}


def log_event(stage, status, message, extra=None):
    record = {
        "event_ts": datetime.now(timezone.utc).isoformat(),
        "stage": stage,
        "status": status,
        "message": message,
        "run_id": RUN_ID,
    }
    if extra:
        record.update(extra)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")


def add_issue(issues, dataset, check_type, severity, issue_count, description, file_path=None, phase="initial"):
    issues.append({
        "phase": phase,
        "dataset": dataset,
        "check_type": check_type,
        "severity": severity,
        "issue_count": int(issue_count),
        "description": description,
        "file_path": str(file_path) if file_path else None,
    })


def add_fix(fixes, dataset, action, rows_before=None, rows_after=None, details=None, file_path=None):
    fixes.append({
        "dataset": dataset,
        "action": action,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "details": details,
        "file_path": str(file_path) if file_path else None,
    })


def latest_match(base_dir: Path, pattern: str):
    if not base_dir.exists():
        return None
    matches = list(base_dir.rglob(pattern))
    if not matches:
        return None
    return max(matches, key=lambda p: p.stat().st_mtime)


def missing_columns(df, expected_columns):
    return [c for c in expected_columns if c not in df.columns]


def duplicate_count(df, subset=None):
    return int(df.duplicated(subset=subset).sum())


def null_summary(df):
    result = {}
    for c in df.columns:
        n = int(df[c].isna().sum())
        if n > 0:
            result[c] = n
    return result


def invalid_numeric_range_count(df, column, min_value=None, max_value=None):
    if column not in df.columns:
        return None
    s = pd.to_numeric(df[column], errors="coerce")
    mask = pd.Series(False, index=df.index)
    if min_value is not None:
        mask = mask | (s < min_value)
    if max_value is not None:
        mask = mask | (s > max_value)
    return int(mask.fillna(False).sum())


def invalid_membership_count(df, column, allowed_values):
    if column not in df.columns:
        return None
    return int((~df[column].isin(allowed_values)).sum())


def invalid_timestamp_count(df, column, unit=None):
    if column not in df.columns:
        return None
    parsed = pd.to_datetime(df[column], errors="coerce", unit=unit)
    return int(parsed.isna().sum())


def build_dataset_summary(dataset_name, df, issues, phase):
    dataset_issues = [x for x in issues if x["dataset"] == dataset_name and x["phase"] == phase]
    error_count = sum(x["issue_count"] for x in dataset_issues if x["severity"] == "error")
    warning_count = sum(x["issue_count"] for x in dataset_issues if x["severity"] == "warning")
    return {
        "phase": phase,
        "dataset": dataset_name,
        "rows": int(len(df)) if df is not None else 0,
        "columns": int(len(df.columns)) if df is not None else 0,
        "errors": int(error_count),
        "warnings": int(warning_count),
        "status": "PASS" if error_count == 0 else "FAIL",
    }


def discover_files(raw_root, bronze_root):
    return {
        "events_csv": latest_match(raw_root, "events.csv"),
        "category_tree_csv": latest_match(raw_root, "category_tree.csv"),
        "item_properties_part1_csv": latest_match(raw_root, "item_properties_part1.csv"),
        "item_properties_part2_csv": latest_match(raw_root, "item_properties_part2.csv"),
        "products_raw_json": latest_match(raw_root, "products_raw.json"),
        "categories_raw_json": latest_match(raw_root, "categories_raw.json"),
        "products_parquet": latest_match(bronze_root, "products.parquet"),
        "categories_parquet": latest_match(bronze_root, "categories.parquet"),
    }


def print_discovered_files(files, raw_root, bronze_root):
    print("PROJECT_ROOT:", PROJECT_ROOT)
    print("RAW_ROOT:", raw_root)
    print("BRONZE_ROOT:", bronze_root)
    print("\nDiscovered files:")
    for k, v in files.items():
        print(f"- {k}: {v}")


def validate_file_presence(files, issues, phase):
    required = [
        "events_csv",
        "category_tree_csv",
        "item_properties_part1_csv",
        "item_properties_part2_csv",
        "products_raw_json",
        "categories_raw_json",
    ]
    for key in required:
        if files.get(key) is None:
            add_issue(
                issues,
                dataset=key,
                check_type="file_presence",
                severity="error",
                issue_count=1,
                description=f"Required file not found for {key}. Check PROJECT_ROOT or run ingestion first.",
                phase=phase,
            )


def validate_retailrocket(files, issues, summaries, phase):
    if not all([
        files["events_csv"],
        files["category_tree_csv"],
        files["item_properties_part1_csv"],
        files["item_properties_part2_csv"],
    ]):
        return

    events = pd.read_csv(files["events_csv"])
    category_tree = pd.read_csv(files["category_tree_csv"])
    item_properties_1 = pd.read_csv(files["item_properties_part1_csv"])
    item_properties_2 = pd.read_csv(files["item_properties_part2_csv"])
    item_properties = pd.concat([item_properties_1, item_properties_2], ignore_index=True)

    exp_events = ["timestamp", "visitorid", "event", "itemid", "transactionid"]
    exp_category = ["categoryid", "parentid"]
    exp_item_props = ["timestamp", "itemid", "property", "value"]

    mc = missing_columns(events, exp_events)
    if mc:
        add_issue(issues, "retailrocket_events", "schema", "error", len(mc), f"Missing columns: {mc}", files["events_csv"], phase)
    for col, cnt in null_summary(events).items():
        add_issue(issues, "retailrocket_events", "missing_values", "warning", cnt, f"Nulls in {col}", files["events_csv"], phase)
    dups = duplicate_count(events, subset=["timestamp", "visitorid", "event", "itemid"])
    if dups > 0:
        add_issue(issues, "retailrocket_events", "duplicates", "warning", dups, "Duplicate rows", files["events_csv"], phase)
    bad_event = invalid_membership_count(events, "event", ALLOWED_EVENT_TYPES)
    if bad_event and bad_event > 0:
        add_issue(issues, "retailrocket_events", "domain", "error", bad_event, "Invalid event values", files["events_csv"], phase)
    bad_ts = invalid_timestamp_count(events, "timestamp", unit="ms")
    if bad_ts and bad_ts > 0:
        add_issue(issues, "retailrocket_events", "format", "error", bad_ts, "Invalid timestamps", files["events_csv"], phase)
    summaries.append(build_dataset_summary("retailrocket_events", events, issues, phase))

    mc = missing_columns(category_tree, exp_category)
    if mc:
        add_issue(issues, "retailrocket_category_tree", "schema", "error", len(mc), f"Missing columns: {mc}", files["category_tree_csv"], phase)
    for col, cnt in null_summary(category_tree).items():
        add_issue(issues, "retailrocket_category_tree", "missing_values", "warning", cnt, f"Nulls in {col}", files["category_tree_csv"], phase)
    dups = duplicate_count(category_tree, subset=["categoryid", "parentid"])
    if dups > 0:
        add_issue(issues, "retailrocket_category_tree", "duplicates", "warning", dups, "Duplicate rows", files["category_tree_csv"], phase)
    summaries.append(build_dataset_summary("retailrocket_category_tree", category_tree, issues, phase))

    mc = missing_columns(item_properties, exp_item_props)
    if mc:
        add_issue(issues, "retailrocket_item_properties", "schema", "error", len(mc), f"Missing columns: {mc}", str(files["item_properties_part1_csv"]), phase)
    for col, cnt in null_summary(item_properties).items():
        add_issue(issues, "retailrocket_item_properties", "missing_values", "warning", cnt, f"Nulls in {col}", str(files["item_properties_part1_csv"]), phase)
    dups = duplicate_count(item_properties, subset=["timestamp", "itemid", "property", "value"])
    if dups > 0:
        add_issue(issues, "retailrocket_item_properties", "duplicates", "warning", dups, "Duplicate rows", str(files["item_properties_part1_csv"]), phase)
    bad_ts = invalid_timestamp_count(item_properties, "timestamp", unit="ms")
    if bad_ts and bad_ts > 0:
        add_issue(issues, "retailrocket_item_properties", "format", "error", bad_ts, "Invalid timestamps", str(files["item_properties_part1_csv"]), phase)
    summaries.append(build_dataset_summary("retailrocket_item_properties", item_properties, issues, phase))


def validate_dummyjson(files, issues, summaries, phase):
    if not all([files["products_raw_json"], files["categories_raw_json"]]):
        return

    with open(files["products_raw_json"], "r", encoding="utf-8") as f:
        products_payload = json.load(f)

    with open(files["categories_raw_json"], "r", encoding="utf-8") as f:
        categories_payload = json.load(f)

    if "products" not in products_payload:
        add_issue(issues, "dummyjson_products_raw", "schema", "error", 1, "Missing 'products' key", files["products_raw_json"], phase)
        products_df = pd.DataFrame()
    else:
        products_df = pd.json_normalize(products_payload["products"], sep="_")

    if isinstance(categories_payload, list):
        if len(categories_payload) > 0 and isinstance(categories_payload[0], dict):
            categories_df = pd.json_normalize(categories_payload, sep="_")
        else:
            categories_df = pd.DataFrame({"category": categories_payload})
    else:
        categories_df = pd.DataFrame()
        add_issue(issues, "dummyjson_categories_raw", "schema", "error", 1, "categories_raw.json must be a list", files["categories_raw_json"], phase)

    exp_products = ["id", "title", "category", "price"]

    mc = missing_columns(products_df, exp_products)
    if mc:
        add_issue(issues, "dummyjson_products_raw", "schema", "error", len(mc), f"Missing columns: {mc}", files["products_raw_json"], phase)
    for col, cnt in null_summary(products_df).items():
        add_issue(issues, "dummyjson_products_raw", "missing_values", "warning", cnt, f"Nulls in {col}", files["products_raw_json"], phase)
    dups = duplicate_count(products_df, subset=["id"]) if "id" in products_df.columns else 0
    if dups > 0:
        add_issue(issues, "dummyjson_products_raw", "duplicates", "error", dups, "Duplicate product IDs", files["products_raw_json"], phase)
    bad_price = invalid_numeric_range_count(products_df, "price", min_value=0)
    if bad_price and bad_price > 0:
        add_issue(issues, "dummyjson_products_raw", "range", "error", bad_price, "Negative prices", files["products_raw_json"], phase)
    bad_rating = invalid_numeric_range_count(products_df, "rating", min_value=1, max_value=5)
    if bad_rating and bad_rating > 0:
        add_issue(issues, "dummyjson_products_raw", "range", "warning", bad_rating, "Ratings outside 1-5", files["products_raw_json"], phase)
    bad_stock = invalid_numeric_range_count(products_df, "stock", min_value=0)
    if bad_stock and bad_stock > 0:
        add_issue(issues, "dummyjson_products_raw", "range", "error", bad_stock, "Negative stock", files["products_raw_json"], phase)
    summaries.append(build_dataset_summary("dummyjson_products_raw", products_df, issues, phase))

    mc = missing_columns(categories_df, ["category"])
    if mc:
        add_issue(issues, "dummyjson_categories_raw", "schema", "error", len(mc), f"Missing columns: {mc}", files["categories_raw_json"], phase)
    for col, cnt in null_summary(categories_df).items():
        add_issue(issues, "dummyjson_categories_raw", "missing_values", "warning", cnt, f"Nulls in {col}", files["categories_raw_json"], phase)
    dups = duplicate_count(categories_df, subset=["category"]) if "category" in categories_df.columns else 0
    if dups > 0:
        add_issue(issues, "dummyjson_categories_raw", "duplicates", "warning", dups, "Duplicate categories", files["categories_raw_json"], phase)
    summaries.append(build_dataset_summary("dummyjson_categories_raw", categories_df, issues, phase))


def validate_bronze(files, issues, summaries, phase):
    for key in ["products_parquet", "categories_parquet"]:
        path = files.get(key)
        if path is None:
            continue

        df = pd.read_parquet(path)
        required_meta = ["source_system", "source_file", "batch_id", "ingestion_ts"]

        mc = missing_columns(df, required_meta)
        if mc:
            add_issue(issues, key, "schema", "warning", len(mc), f"Missing bronze metadata columns: {mc}", path, phase)
        if len(df) == 0:
            add_issue(issues, key, "completeness", "error", 1, "Empty bronze file", path, phase)

        summaries.append(build_dataset_summary(key, df, issues, phase))


def run_validation_cycle(files, phase):
    issues = []
    summaries = []

    validate_file_presence(files, issues, phase)
    validate_retailrocket(files, issues, summaries, phase)
    validate_dummyjson(files, issues, summaries, phase)
    validate_bronze(files, issues, summaries, phase)

    issues_df = pd.DataFrame(issues)
    summary_df = pd.DataFrame(summaries)

    if issues_df.empty:
        issues_df = pd.DataFrame(columns=["phase", "dataset", "check_type", "severity", "issue_count", "description", "file_path"])
    if summary_df.empty:
        summary_df = pd.DataFrame(columns=["phase", "dataset", "rows", "columns", "errors", "warnings", "status"])

    overall_status = "PASS" if len(issues_df[issues_df["severity"] == "error"]) == 0 else "FAIL"

    return issues_df, summary_df, overall_status


def ensure_columns(df, columns):
    for col in columns:
        if col not in df.columns:
            df[col] = pd.NA
    return df


def save_json(path, payload):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)


def fix_events(src_path, dst_path, fixes):
    df = pd.read_csv(src_path)
    rows_before = len(df)

    expected = ["timestamp", "visitorid", "event", "itemid", "transactionid"]
    missing = [c for c in expected if c not in df.columns]
    df = ensure_columns(df, expected)
    if missing:
        add_fix(fixes, "retailrocket_events", "add_missing_columns", rows_before, len(df), f"Added columns: {missing}", src_path)

    if "event" in df.columns:
        df["event"] = df["event"].astype(str).str.strip().str.lower()
        before = len(df)
        df = df[df["event"].isin(ALLOWED_EVENT_TYPES) | df["event"].isna()]
        add_fix(fixes, "retailrocket_events", "remove_invalid_event_values", before, len(df), "Kept only allowed event types", src_path)

    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
        before = len(df)
        df = df[df["timestamp"].notna()]
        add_fix(fixes, "retailrocket_events", "drop_invalid_timestamps", before, len(df), "Removed rows with invalid timestamp", src_path)

    for col in ["visitorid", "itemid", "transactionid"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    before = len(df)
    df = df.drop_duplicates(subset=["timestamp", "visitorid", "event", "itemid"])
    add_fix(fixes, "retailrocket_events", "drop_duplicates", before, len(df), "Removed duplicate event rows", src_path)

    before = len(df)
    df = df.dropna(subset=["timestamp", "visitorid", "event", "itemid"])
    add_fix(fixes, "retailrocket_events", "drop_missing_required_rows", before, len(df), "Dropped rows missing required fields", src_path)

    df.to_csv(dst_path, index=False)
    return dst_path


def fix_category_tree(src_path, dst_path, fixes):
    df = pd.read_csv(src_path)
    rows_before = len(df)

    expected = ["categoryid", "parentid"]
    missing = [c for c in expected if c not in df.columns]
    df = ensure_columns(df, expected)
    if missing:
        add_fix(fixes, "retailrocket_category_tree", "add_missing_columns", rows_before, len(df), f"Added columns: {missing}", src_path)

    for col in ["categoryid", "parentid"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    before = len(df)
    df = df.dropna(subset=["categoryid"])
    add_fix(fixes, "retailrocket_category_tree", "drop_missing_categoryid", before, len(df), "Dropped rows with missing categoryid", src_path)

    before = len(df)
    df = df.drop_duplicates(subset=["categoryid", "parentid"])
    add_fix(fixes, "retailrocket_category_tree", "drop_duplicates", before, len(df), "Removed duplicate category rows", src_path)

    df.to_csv(dst_path, index=False)
    return dst_path


def fix_item_properties(part1_path, part2_path, dst_part1_path, dst_part2_path, fixes):
    df1 = pd.read_csv(part1_path)
    df2 = pd.read_csv(part2_path)
    df = pd.concat([df1, df2], ignore_index=True)
    rows_before = len(df)

    expected = ["timestamp", "itemid", "property", "value"]
    missing = [c for c in expected if c not in df.columns]
    df = ensure_columns(df, expected)
    if missing:
        add_fix(fixes, "retailrocket_item_properties", "add_missing_columns", rows_before, len(df), f"Added columns: {missing}", part1_path)

    df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
    df["itemid"] = pd.to_numeric(df["itemid"], errors="coerce")
    df["property"] = df["property"].astype("string").str.strip()
    df["value"] = df["value"].astype("string").str.strip()

    before = len(df)
    df = df.dropna(subset=["timestamp", "itemid", "property", "value"])
    add_fix(fixes, "retailrocket_item_properties", "drop_missing_required_rows", before, len(df), "Dropped rows with invalid required values", part1_path)

    before = len(df)
    df = df.drop_duplicates(subset=["timestamp", "itemid", "property", "value"])
    add_fix(fixes, "retailrocket_item_properties", "drop_duplicates", before, len(df), "Removed duplicate property rows", part1_path)

    split_index = len(df) // 2
    df.iloc[:split_index].to_csv(dst_part1_path, index=False)
    df.iloc[split_index:].to_csv(dst_part2_path, index=False)
    return dst_part1_path, dst_part2_path


def fix_products_raw_json(src_path, dst_path, fixes):
    with open(src_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    products = payload.get("products", [])
    if not isinstance(products, list):
        products = []

    df = pd.json_normalize(products, sep="_") if len(products) > 0 else pd.DataFrame()
    rows_before = len(df)

    expected = ["id", "title", "category", "price", "rating", "stock"]
    missing = [c for c in expected if c not in df.columns]
    df = ensure_columns(df, expected)
    if missing:
        add_fix(fixes, "dummyjson_products_raw", "add_missing_columns", rows_before, len(df), f"Added columns: {missing}", src_path)

    for col in ["id", "price", "rating", "stock"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if "title" in df.columns:
        df["title"] = df["title"].astype("string").str.strip()
    if "category" in df.columns:
        df["category"] = df["category"].astype("string").str.strip()

    before = len(df)
    df = df.dropna(subset=["id", "title", "category", "price"])
    add_fix(fixes, "dummyjson_products_raw", "drop_missing_required_rows", before, len(df), "Dropped rows missing id/title/category/price", src_path)

    before = len(df)
    df = df.drop_duplicates(subset=["id"])
    add_fix(fixes, "dummyjson_products_raw", "drop_duplicate_ids", before, len(df), "Kept first product per id", src_path)

    neg_price = int((df["price"] < 0).fillna(False).sum())
    df["price"] = df["price"].clip(lower=0)
    add_fix(fixes, "dummyjson_products_raw", "fix_negative_prices", len(df), len(df), f"Clipped {neg_price} negative prices to 0", src_path)

    if "rating" in df.columns:
        bad_rating = int(((df["rating"] < 1) | (df["rating"] > 5)).fillna(False).sum())
        df["rating"] = df["rating"].clip(lower=1, upper=5)
        add_fix(fixes, "dummyjson_products_raw", "clip_ratings", len(df), len(df), f"Clipped {bad_rating} rating values into 1-5", src_path)

    if "stock" in df.columns:
        neg_stock = int((df["stock"] < 0).fillna(False).sum())
        df["stock"] = df["stock"].clip(lower=0)
        add_fix(fixes, "dummyjson_products_raw", "fix_negative_stock", len(df), len(df), f"Clipped {neg_stock} stock values to 0", src_path)

    clean_payload = {"products": df.where(pd.notna(df), None).to_dict(orient="records")}
    save_json(dst_path, clean_payload)
    return dst_path


def fix_categories_raw_json(src_path, dst_path, fixes):
    with open(src_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    if isinstance(payload, list):
        if len(payload) > 0 and isinstance(payload[0], dict):
            df = pd.json_normalize(payload, sep="_")
        else:
            df = pd.DataFrame({"category": payload})
    else:
        df = pd.DataFrame({"category": []})

    rows_before = len(df)
    df = ensure_columns(df, ["category"])
    df["category"] = df["category"].astype("string").str.strip()

    before = len(df)
    df = df.dropna(subset=["category"])
    add_fix(fixes, "dummyjson_categories_raw", "drop_missing_category", before, len(df), "Dropped null categories", src_path)

    before = len(df)
    df = df[df["category"] != ""]
    add_fix(fixes, "dummyjson_categories_raw", "drop_blank_category", before, len(df), "Dropped blank categories", src_path)

    before = len(df)
    df = df.drop_duplicates(subset=["category"])
    add_fix(fixes, "dummyjson_categories_raw", "drop_duplicates", before, len(df), "Removed duplicate categories", src_path)

    save_json(dst_path, df["category"].tolist())
    return dst_path


def fix_bronze_parquet(src_path, dst_path, dataset_name, fixes):
    df = pd.read_parquet(src_path)
    rows_before = len(df)

    required_meta = ["source_system", "source_file", "batch_id", "ingestion_ts"]
    for col in required_meta:
        if col not in df.columns:
            if col == "source_system":
                df[col] = "unknown"
            elif col == "source_file":
                df[col] = src_path.name
            elif col == "batch_id":
                df[col] = RUN_ID
            elif col == "ingestion_ts":
                df[col] = RUN_TS

    add_fix(fixes, dataset_name, "add_missing_bronze_metadata", rows_before, len(df), "Ensured bronze metadata columns exist", src_path)

    df.to_parquet(dst_path, index=False)
    return dst_path


def remediate_files(files):
    fixes = []

    remediated = dict(files)

    if files.get("events_csv"):
        remediated["events_csv"] = fix_events(
            files["events_csv"],
            REMEDIATED_RAW_ROOT / "events.csv",
            fixes,
        )

    if files.get("category_tree_csv"):
        remediated["category_tree_csv"] = fix_category_tree(
            files["category_tree_csv"],
            REMEDIATED_RAW_ROOT / "category_tree.csv",
            fixes,
        )

    if files.get("item_properties_part1_csv") and files.get("item_properties_part2_csv"):
        part1, part2 = fix_item_properties(
            files["item_properties_part1_csv"],
            files["item_properties_part2_csv"],
            REMEDIATED_RAW_ROOT / "item_properties_part1.csv",
            REMEDIATED_RAW_ROOT / "item_properties_part2.csv",
            fixes,
        )
        remediated["item_properties_part1_csv"] = part1
        remediated["item_properties_part2_csv"] = part2

    if files.get("products_raw_json"):
        remediated["products_raw_json"] = fix_products_raw_json(
            files["products_raw_json"],
            REMEDIATED_RAW_ROOT / "products_raw.json",
            fixes,
        )

    if files.get("categories_raw_json"):
        remediated["categories_raw_json"] = fix_categories_raw_json(
            files["categories_raw_json"],
            REMEDIATED_RAW_ROOT / "categories_raw.json",
            fixes,
        )

    if files.get("products_parquet"):
        remediated["products_parquet"] = fix_bronze_parquet(
            files["products_parquet"],
            REMEDIATED_BRONZE_ROOT / "products.parquet",
            "products_parquet",
            fixes,
        )

    if files.get("categories_parquet"):
        remediated["categories_parquet"] = fix_bronze_parquet(
            files["categories_parquet"],
            REMEDIATED_BRONZE_ROOT / "categories.parquet",
            "categories_parquet",
            fixes,
        )

    fixes_df = pd.DataFrame(fixes)
    if fixes_df.empty:
        fixes_df = pd.DataFrame(columns=["dataset", "action", "rows_before", "rows_after", "details", "file_path"])

    return remediated, fixes_df


def df_to_table_data(df, max_rows=20):
    if df is None or df.empty:
        return [["No data"]]
    preview = df.head(max_rows).copy()
    preview = preview.fillna("")
    return [list(preview.columns)] + preview.astype(str).values.tolist()


def add_pdf_table(story, title, df, max_rows=20):
    styles = getSampleStyleSheet()
    story.append(Paragraph(title, styles["Heading3"]))
    data = df_to_table_data(df, max_rows=max_rows)
    table = Table(data, repeatRows=1)
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#D9EAD3")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTSIZE", (0, 0), (-1, -1), 8),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
    ]))
    story.append(table)
    story.append(Spacer(1, 12))


def generate_pdf_report(initial_report, final_report, initial_issues_df, final_issues_df, initial_summary_df, final_summary_df, fixes_df):
    styles = getSampleStyleSheet()
    story = []

    story.append(Paragraph("RecoMart Data Quality Report", styles["Title"]))
    story.append(Spacer(1, 12))
    story.append(Paragraph(f"Run ID: {RUN_ID}", styles["Normal"]))
    story.append(Paragraph(f"Generated at: {RUN_TS}", styles["Normal"]))
    story.append(Paragraph(f"Project Root: {PROJECT_ROOT}", styles["Normal"]))
    story.append(Spacer(1, 12))

    story.append(Paragraph("1. Initial Validation", styles["Heading2"]))
    story.append(Paragraph(f"Initial overall status: <b>{initial_report['overall_status']}</b>", styles["Normal"]))
    story.append(Spacer(1, 12))
    add_pdf_table(story, "Initial Dataset Summary", initial_summary_df)
    add_pdf_table(story, "Initial Validation Issues", initial_issues_df)

    story.append(PageBreak())

    story.append(Paragraph("2. Data Fixing Actions", styles["Heading2"]))
    story.append(Paragraph("The following remediation steps were applied automatically to failed or warning-prone datasets.", styles["Normal"]))
    story.append(Spacer(1, 12))
    add_pdf_table(story, "Fix Log", fixes_df)

    story.append(PageBreak())

    story.append(Paragraph("3. Revalidation", styles["Heading2"]))
    story.append(Paragraph(f"Revalidation overall status: <b>{final_report['overall_status']}</b>", styles["Normal"]))
    story.append(Spacer(1, 12))
    add_pdf_table(story, "Revalidated Dataset Summary", final_summary_df)
    add_pdf_table(story, "Revalidation Issues", final_issues_df)

    story.append(PageBreak())

    story.append(Paragraph("4. Before vs After", styles["Heading2"]))
    before_errors = int(initial_issues_df.loc[initial_issues_df["severity"] == "error", "issue_count"].sum()) if not initial_issues_df.empty else 0
    after_errors = int(final_issues_df.loc[final_issues_df["severity"] == "error", "issue_count"].sum()) if not final_issues_df.empty else 0
    before_warnings = int(initial_issues_df.loc[initial_issues_df["severity"] == "warning", "issue_count"].sum()) if not initial_issues_df.empty else 0
    after_warnings = int(final_issues_df.loc[final_issues_df["severity"] == "warning", "issue_count"].sum()) if not final_issues_df.empty else 0

    comparison_df = pd.DataFrame([
        {"metric": "overall_status", "before": initial_report["overall_status"], "after": final_report["overall_status"]},
        {"metric": "error_count", "before": before_errors, "after": after_errors},
        {"metric": "warning_count", "before": before_warnings, "after": after_warnings},
    ])
    add_pdf_table(story, "Validation Comparison", comparison_df)

    story.append(Paragraph("5. Conclusion", styles["Heading2"]))
    conclusion = (
        f"The initial validation run returned <b>{initial_report['overall_status']}</b>. "
        f"After applying automated remediation steps and rerunning validation on the remediated files, "
        f"the final status returned <b>{final_report['overall_status']}</b>. "
        f"The detailed issue logs, fix log, and dataset summaries have been saved alongside this PDF."
    )
    story.append(Paragraph(conclusion, styles["Normal"]))

    doc = SimpleDocTemplate(str(PDF_REPORT_FILE), pagesize=A4)
    doc.build(story)


def main():
    log_event("validation", "started", "Validation started")

    original_files = discover_files(RAW_ROOT, BRONZE_ROOT)
    print_discovered_files(original_files, RAW_ROOT, BRONZE_ROOT)

    initial_issues_df, initial_summary_df, initial_status = run_validation_cycle(original_files, phase="initial")

    initial_issues_df.to_csv(INITIAL_ISSUES_FILE, index=False)
    initial_summary_df.to_csv(INITIAL_SUMMARY_FILE, index=False)

    initial_report = {
        "run_id": RUN_ID,
        "run_ts": RUN_TS,
        "phase": "initial",
        "project_root": str(PROJECT_ROOT),
        "raw_root": str(RAW_ROOT),
        "bronze_root": str(BRONZE_ROOT),
        "discovered_files": {k: (str(v) if v else None) for k, v in original_files.items()},
        "overall_status": initial_status,
        "issues_file": str(INITIAL_ISSUES_FILE),
        "summary_file": str(INITIAL_SUMMARY_FILE),
        "datasets": initial_summary_df.to_dict(orient="records"),
    }

    if initial_status == "FAIL":
        log_event("remediation", "started", "Validation failed; starting remediation")
        remediated_files, fixes_df = remediate_files(original_files)
        fixes_df.to_csv(FIX_LOG_FILE, index=False)

        final_issues_df, final_summary_df, final_status = run_validation_cycle(remediated_files, phase="revalidated")
        final_issues_df.to_csv(FINAL_ISSUES_FILE, index=False)
        final_summary_df.to_csv(FINAL_SUMMARY_FILE, index=False)

        final_report = {
            "run_id": RUN_ID,
            "run_ts": RUN_TS,
            "phase": "revalidated",
            "remediated_raw_root": str(REMEDIATED_RAW_ROOT),
            "remediated_bronze_root": str(REMEDIATED_BRONZE_ROOT),
            "discovered_files": {k: (str(v) if v else None) for k, v in remediated_files.items()},
            "overall_status": final_status,
            "issues_file": str(FINAL_ISSUES_FILE),
            "summary_file": str(FINAL_SUMMARY_FILE),
            "fix_log_file": str(FIX_LOG_FILE),
            "datasets": final_summary_df.to_dict(orient="records"),
        }
    else:
        log_event("remediation", "skipped", "Initial validation passed; no remediation required")
        fixes_df = pd.DataFrame([{
            "dataset": "all",
            "action": "no_action_required",
            "rows_before": None,
            "rows_after": None,
            "details": "Initial validation passed; remediation skipped",
            "file_path": None,
        }])
        fixes_df.to_csv(FIX_LOG_FILE, index=False)

        final_issues_df = initial_issues_df.copy()
        final_summary_df = initial_summary_df.copy()
        final_status = initial_status

        final_report = {
            "run_id": RUN_ID,
            "run_ts": RUN_TS,
            "phase": "revalidated",
            "overall_status": final_status,
            "issues_file": str(FINAL_ISSUES_FILE),
            "summary_file": str(FINAL_SUMMARY_FILE),
            "fix_log_file": str(FIX_LOG_FILE),
            "datasets": final_summary_df.to_dict(orient="records"),
        }

        final_issues_df.to_csv(FINAL_ISSUES_FILE, index=False)
        final_summary_df.to_csv(FINAL_SUMMARY_FILE, index=False)

    full_report = {
        "run_id": RUN_ID,
        "run_ts": RUN_TS,
        "project_root": str(PROJECT_ROOT),
        "initial_validation": initial_report,
        "revalidation": final_report,
        "pdf_report_file": str(PDF_REPORT_FILE),
    }

    with open(REPORT_FILE, "w", encoding="utf-8") as f:
        json.dump(full_report, f, indent=2)

    generate_pdf_report(
        initial_report=initial_report,
        final_report=final_report,
        initial_issues_df=initial_issues_df,
        final_issues_df=final_issues_df,
        initial_summary_df=initial_summary_df,
        final_summary_df=final_summary_df,
        fixes_df=fixes_df,
    )

    print("\nInitial status:", initial_report["overall_status"])
    print("Final status:", final_report["overall_status"])
    print("Initial issues file:", INITIAL_ISSUES_FILE)
    print("Initial summary file:", INITIAL_SUMMARY_FILE)
    print("Revalidation issues file:", FINAL_ISSUES_FILE)
    print("Revalidation summary file:", FINAL_SUMMARY_FILE)
    print("Fix log file:", FIX_LOG_FILE)
    print("JSON report file:", REPORT_FILE)
    print("PDF report file:", PDF_REPORT_FILE)

    log_event("validation", "completed", "Validation workflow completed", {
        "initial_status": initial_report["overall_status"],
        "final_status": final_report["overall_status"],
        "pdf_report_file": str(PDF_REPORT_FILE),
    })

    return {
        "original_files": original_files,
        "initial_issues_df": initial_issues_df,
        "initial_summary_df": initial_summary_df,
        "final_issues_df": final_issues_df,
        "final_summary_df": final_summary_df,
        "fixes_df": fixes_df,
        "report": full_report,
    }


results = main()


ModuleNotFoundError: No module named 'reportlab'